# Premier League: all historik → 2023/24 holdout

Denna notebook tränar på **all tillgänglig Premier League-historik före 2023/24** (2000/01–2022/23) och använder 2023/24 som helt orörd testperiod.

Upplägget är strikt tidsbaserat:

- En rad per lag och match, ur lagets perspektiv.
- `Target` är `Vinst`, `Oavgjort` eller `Förlust`.
- Alla features är pre-match och använder endast tidigare matcher.
- Fem CV-folds validerar på 2018/19–2022/23; varje fold tränas på all äldre historik.
- Numeriska features imputeras, korrelationsrensas och skalas inuti varje fold.
- Logistisk regression jämförs med en lätt regulariserad boostingmodell.
- Native XGBoost används när dess runtime fungerar; annars används en tydligt märkt sklearn-fallback.
- Modellval görs endast på historisk CV. Teståret påverkar inga val.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize

here = Path.cwd().resolve()
root = next(path for path in [here, *here.parents] if (path / "src" / "data_loader.py").exists())
sys.path.insert(0, str(root / "src"))

from data_loader import load_raw_data
from feature_engineering import engineer_features

try:
    from xgboost import XGBClassifier
    NATIVE_XGBOOST = True
    BOOSTING_MODEL_NAME = "XGBoost light"
except Exception as error:
    XGBOOST_IMPORT_ERROR = str(error)
    XGBClassifier = None
    NATIVE_XGBOOST = False
    BOOSTING_MODEL_NAME = "Sklearn HistGradientBoosting (XGBoost fallback)"

LEAGUE = "Premier League"
TEST_SEASON = "2023/24"
CV_VALIDATION_SEASONS = ["2018/19", "2019/20", "2020/21", "2021/22", "2022/23"]
CLASS_ORDER = ["Förlust", "Oavgjort", "Vinst"]
TARGET_TO_CODE = {label: index for index, label in enumerate(CLASS_ORDER)}
DATA_DIR = root / "data"
DATA_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 2026

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Ladda Premier League och skapa pre-match-features

Featuremotorn bearbetar hela historiken kronologiskt. Resultat från en match får uppdatera laghistoriken först efter att dagens samtliga pre-match-rader skapats.

In [2]:
raw = load_raw_data()
premier_league = raw.loc[
    (raw["League"] == LEAGUE) & (raw["Season"] <= TEST_SEASON)
].copy()
premier_league["MatchDate"] = pd.to_datetime(premier_league["MatchDate"])
premier_league = premier_league.sort_values(["MatchDate", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

engineered = engineer_features(premier_league)

print(f"Matcher: {len(premier_league):,}")
print(f"Säsonger: {premier_league['Season'].nunique()}")
print(f"Period: {premier_league.MatchDate.min():%Y-%m-%d} – {premier_league.MatchDate.max():%Y-%m-%d}")
print(f"Train-säsonger: {premier_league.loc[premier_league.Season < TEST_SEASON, 'Season'].min()}–{premier_league.loc[premier_league.Season < TEST_SEASON, 'Season'].max()}")
print(f"Test-säsong: {TEST_SEASON}")

Matcher: 9,120
Säsonger: 24
Period: 2000-08-19 – 2024-05-19
Train-säsonger: 2000/01–2022/23
Test-säsong: 2023/24


## 2. Gör datan lagcentrerad

Varje fysisk match blir två lagrader med samma `MatchId`. Rå matchstatistik används endast för att bygga historiska features och tas bort före modellering. Total-, hemma- och bortahistorik beräknas separat och strikt före aktuell match.

In [3]:
def build_team_data(raw_matches, engineered_matches):
    raw_matches = raw_matches.copy()
    raw_matches["MatchDate"] = pd.to_datetime(raw_matches["MatchDate"])
    raw_matches["MatchId"] = (
        raw_matches["League"].astype(str) + "_"
        + raw_matches["MatchDate"].dt.strftime("%Y%m%d") + "_"
        + raw_matches["HomeTeam"] + "_" + raw_matches["AwayTeam"]
    )
    result_map = {
        "home": {"H": "Vinst", "D": "Oavgjort", "A": "Förlust"},
        "away": {"H": "Förlust", "D": "Oavgjort", "A": "Vinst"},
    }
    parts = []
    for side, venue in (("home", "Hemma"), ("away", "Borta")):
        own = "Home" if side == "home" else "Away"
        opponent = "Away" if side == "home" else "Home"
        part = pd.DataFrame({
            "MatchId": raw_matches["MatchId"],
            "Season": raw_matches["Season"],
            "MatchDate": raw_matches["MatchDate"],
            "Team": raw_matches[f"{own}Team"],
            "Opponent": raw_matches[f"{opponent}Team"],
            "Venue": venue,
            "Target": raw_matches["FullTimeResult"].map(result_map[side]),
            "_points": raw_matches["FullTimeResult"].map(
                {"H": 3, "D": 1, "A": 0} if side == "home" else {"H": 0, "D": 1, "A": 3}
            ),
            "_goal_diff": raw_matches[f"FullTime{own}Goals"] - raw_matches[f"FullTime{opponent}Goals"],
            "_shots_for": raw_matches[f"{own}Shots"],
            "_shots_against": raw_matches[f"{opponent}Shots"],
            "_sot_for": raw_matches[f"{own}ShotsOnTarget"],
            "_sot_against": raw_matches[f"{opponent}ShotsOnTarget"],
        })
        parts.append(part)

    long = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["Team", "MatchDate", "MatchId"])
        .reset_index(drop=True)
    )
    metrics = ["points", "goal_diff", "shots_for", "shots_against", "sot_for", "sot_against"]
    records = []
    for _, matches in long.groupby("Team", sort=False):
        histories = {"total": [], "home": [], "away": []}
        for index, row in matches.iterrows():
            values = {}
            for scope in ("total", "home", "away"):
                recent = histories[scope][-5:]
                for metric in metrics:
                    observed = [item[metric] for item in recent if pd.notna(item[metric])]
                    values[f"{metric}_{scope}_l5"] = float(np.sum(observed)) if observed else np.nan
                values[f"ppg_{scope}_l5"] = (
                    values[f"points_{scope}_l5"] / len(recent) if recent else np.nan
                )
            records.append((index, values))
            history_row = {metric: row[f"_{metric}"] for metric in metrics}
            histories["total"].append(history_row)
            histories["home" if row["Venue"] == "Hemma" else "away"].append(history_row)

    rolling = pd.DataFrame({index: values for index, values in records}).T
    long = pd.concat([long, rolling.reindex(long.index)], axis=1)

    engineered_matches = engineered_matches.copy()
    engineered_matches["MatchDate"] = pd.to_datetime(engineered_matches["MatchDate"])
    engineered_matches["MatchId"] = (
        engineered_matches["League"].astype(str) + "_"
        + engineered_matches["MatchDate"].dt.strftime("%Y%m%d") + "_"
        + engineered_matches["HomeTeam"] + "_" + engineered_matches["AwayTeam"]
    )
    perspectives = []
    for side, opponent in (("home", "away"), ("away", "home")):
        is_home = side == "home"
        perspectives.append(pd.DataFrame({
            "MatchId": engineered_matches["MatchId"],
            "Team": engineered_matches["HomeTeam" if is_home else "AwayTeam"],
            "team_position": engineered_matches[f"{side}_position"],
            "opponent_position": engineered_matches[f"{opponent}_position"],
            "team_elo": engineered_matches[f"{side}_elo"],
            "opponent_elo": engineered_matches[f"{opponent}_elo"],
            "team_elo_expected": engineered_matches[f"{side}_elo_expected"],
            "expected_team_goals": engineered_matches[f"expected_{side}_goals"],
            "expected_opponent_goals": engineered_matches[f"expected_{opponent}_goals"],
            "poisson_prob_win": engineered_matches[f"poisson_prob_{side}"],
            "poisson_prob_loss": engineered_matches[f"poisson_prob_{opponent}"],
        }))
    perspective = pd.concat(perspectives, ignore_index=True)
    long = long.merge(perspective, on=["MatchId", "Team"], validate="one_to_one")
    long["position_advantage"] = long["opponent_position"] - long["team_position"]
    long["elo_difference"] = long["team_elo"] - long["opponent_elo"]
    long["expected_goal_difference"] = long["expected_team_goals"] - long["expected_opponent_goals"]

    internal = [column for column in long if column.startswith("_")]
    return (
        long.drop(columns=internal)
        .sort_values(["MatchDate", "MatchId", "Team"])
        .reset_index(drop=True)
    )


team_data = build_team_data(premier_league, engineered)
train_data = team_data.loc[team_data["Season"] < TEST_SEASON].copy()
test_data = team_data.loc[team_data["Season"] == TEST_SEASON].copy()

assert set(train_data["MatchId"]).isdisjoint(test_data["MatchId"])
assert len(test_data) == 760
print(f"Train: {len(train_data):,} lagrader, {train_data.Season.nunique()} säsonger")
print(f"Test: {len(test_data):,} lagrader, {test_data.Season.nunique()} säsong")
print(train_data.groupby("Season").size().tail(8))

Train: 17,480 lagrader, 23 säsonger
Test: 760 lagrader, 1 säsong
Season
2015/16    760
2016/17    760
2017/18    760
2018/19    760
2019/20    760
2020/21    760
2021/22    760
2022/23    760
dtype: int64


## 3. Pipeline och säsongsbaserad cross-validation

Valideringssäsongerna är hela och sammanhängande. Exempel: folden för 2021/22 tränar på 2000/01–2020/21. Det motsvarar hur modellen faktiskt skulle användas inför en ny säsong.

In [4]:
class CorrelationPruner(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        correlation = np.nan_to_num(np.corrcoef(values, rowvar=False), nan=0.0)
        kept = []
        for index in range(values.shape[1]):
            if all(abs(correlation[index, previous]) < self.threshold for previous in kept):
                kept.append(index)
        self.kept_indices_ = np.asarray(kept, dtype=int)
        self.n_features_in_ = values.shape[1]
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.kept_indices_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)[self.kept_indices_]


FEATURE_SETS = {
    "form": [
        "ppg_total_l5", "ppg_home_l5", "ppg_away_l5",
        "goal_diff_total_l5", "goal_diff_home_l5", "goal_diff_away_l5",
        "shots_for_total_l5", "shots_against_total_l5",
        "sot_for_total_l5", "sot_against_total_l5",
    ],
    "form_strength": [
        "ppg_total_l5", "ppg_home_l5", "ppg_away_l5",
        "goal_diff_total_l5", "goal_diff_home_l5", "goal_diff_away_l5",
        "shots_for_total_l5", "shots_against_total_l5",
        "sot_for_total_l5", "sot_against_total_l5",
        "position_advantage", "elo_difference",
    ],
    "form_strength_model": [
        "ppg_total_l5", "ppg_home_l5", "ppg_away_l5",
        "goal_diff_total_l5", "goal_diff_home_l5", "goal_diff_away_l5",
        "shots_for_total_l5", "shots_against_total_l5",
        "sot_for_total_l5", "sot_against_total_l5",
        "position_advantage", "elo_difference", "team_elo_expected",
        "expected_goal_difference", "poisson_prob_win", "poisson_prob_loss",
    ],
}
CATEGORICAL_COLUMNS = ["Venue"]

folds = []
for validation_season in CV_VALIDATION_SEASONS:
    train_mask = train_data["Season"] < validation_season
    validation_mask = train_data["Season"] == validation_season
    assert train_mask.any() and validation_mask.any()
    assert train_data.loc[train_mask, "MatchDate"].max() < train_data.loc[validation_mask, "MatchDate"].min()
    folds.append((validation_season, train_mask, validation_mask))
assert len(folds) == 5


def preprocessing_pipeline(numeric_columns):
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("correlation_pruner", CorrelationPruner(threshold=0.90)),
            ("scaler", StandardScaler()),
        ]), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CATEGORICAL_COLUMNS),
    ])


def probabilities_in_order(model, frame, columns, encoded_target=False):
    probabilities = model.predict_proba(frame[columns])
    if encoded_target:
        return probabilities
    fitted_classes = list(model.named_steps["model"].classes_)
    return np.column_stack([probabilities[:, fitted_classes.index(label)] for label in CLASS_ORDER])


def macro_roc(target, probabilities):
    binary_target = label_binarize(target, classes=CLASS_ORDER)
    grid = np.linspace(0, 1, 501)
    interpolated = []
    class_gini = {}
    for index, label in enumerate(CLASS_ORDER):
        fpr, tpr, _ = roc_curve(binary_target[:, index], probabilities[:, index])
        interpolated.append(np.interp(grid, fpr, tpr))
        class_gini[label] = 2 * roc_auc_score(binary_target[:, index], probabilities[:, index]) - 1
    mean_tpr = np.mean(interpolated, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    return grid, mean_tpr, float(np.mean(list(class_gini.values()))), class_gini


def metric_row(model_name, split_name, target, probabilities):
    predicted = np.asarray(CLASS_ORDER)[probabilities.argmax(axis=1)]
    _, _, gini, class_gini = macro_roc(target, probabilities)
    return {
        "Model": model_name,
        "Split": split_name,
        "Rows": len(target),
        "Accuracy": accuracy_score(target, predicted),
        "LogLoss": log_loss(target, probabilities, labels=CLASS_ORDER),
        "MacroGini": gini,
        **{f"Gini_{label}": class_gini[label] for label in CLASS_ORDER},
    }

## 4. Logistisk regression

Hyperparametrar väljs efter lägst sammanlagd OOF-log-loss över de fem valideringssäsongerna. Klassvikt testas, men teståret används inte.

In [5]:
logistic_rows = []
logistic_oof = {}
for feature_name, numeric_columns in FEATURE_SETS.items():
    model_columns = CATEGORICAL_COLUMNS + numeric_columns
    for c_value in (0.003, 0.01, 0.03, 0.1, 0.3, 1.0):
        for class_weight in (None, "balanced"):
            fold_predictions = []
            for validation_season, train_mask, validation_mask in folds:
                model = Pipeline([
                    ("preprocessing", preprocessing_pipeline(numeric_columns)),
                    ("model", LogisticRegression(
                        C=c_value, class_weight=class_weight, max_iter=5_000,
                    )),
                ])
                model.fit(train_data.loc[train_mask, model_columns], train_data.loc[train_mask, "Target"])
                probabilities = probabilities_in_order(
                    model, train_data.loc[validation_mask], model_columns
                )
                fold_predictions.append(pd.DataFrame({
                    "Season": validation_season,
                    "Target": train_data.loc[validation_mask, "Target"].to_numpy(),
                    "ProbabilityLoss": probabilities[:, 0],
                    "ProbabilityDraw": probabilities[:, 1],
                    "ProbabilityWin": probabilities[:, 2],
                }))
            oof = pd.concat(fold_predictions, ignore_index=True)
            probabilities = oof[["ProbabilityLoss", "ProbabilityDraw", "ProbabilityWin"]].to_numpy()
            predicted = np.asarray(CLASS_ORDER)[probabilities.argmax(axis=1)]
            _, _, gini, _ = macro_roc(oof["Target"], probabilities)
            key = (feature_name, c_value, class_weight)
            logistic_oof[key] = oof
            logistic_rows.append({
                "FeatureSet": feature_name,
                "C": c_value,
                "ClassWeight": str(class_weight),
                "CV_LogLoss": log_loss(oof["Target"], probabilities, labels=CLASS_ORDER),
                "CV_Accuracy": accuracy_score(oof["Target"], predicted),
                "CV_MacroGini": gini,
            })

logistic_tuning = pd.DataFrame(logistic_rows).sort_values(
    ["CV_LogLoss", "CV_MacroGini"], ascending=[True, False]
).reset_index(drop=True)
logistic_best = logistic_tuning.iloc[0]
logistic_class_weight = None if logistic_best["ClassWeight"] == "None" else "balanced"
logistic_key = (logistic_best["FeatureSet"], float(logistic_best["C"]), logistic_class_weight)
logistic_features = FEATURE_SETS[logistic_key[0]]
logistic_columns = CATEGORICAL_COLUMNS + logistic_features
logistic_best_oof = logistic_oof[logistic_key]
logistic_oof_probabilities = logistic_best_oof[[
    "ProbabilityLoss", "ProbabilityDraw", "ProbabilityWin",
]].to_numpy()

logistic_model = Pipeline([
    ("preprocessing", preprocessing_pipeline(logistic_features)),
    ("model", LogisticRegression(
        C=logistic_key[1], class_weight=logistic_key[2], max_iter=5_000,
    )),
])
logistic_model.fit(train_data[logistic_columns], train_data["Target"])
logistic_train_probabilities = probabilities_in_order(logistic_model, train_data, logistic_columns)
logistic_test_probabilities = probabilities_in_order(logistic_model, test_data, logistic_columns)
logistic_confidence_threshold = float(np.median(logistic_oof_probabilities.max(axis=1)))

print("Vald logistisk konfiguration:", logistic_key)
print(f"OOF-rader: {len(logistic_best_oof):,}")
print(f"Säkerhetsgräns: {logistic_confidence_threshold:.3f}")
display(logistic_tuning.head(10).round(4))

Vald logistisk konfiguration: ('form_strength_model', 0.003, None)
OOF-rader: 3,800
Säkerhetsgräns: 0.518


,FeatureSet,C,ClassWeight,CV_LogLoss,CV_Accuracy,CV_MacroGini
0,form_strength_model,0.003,None,0.9661,0.5437,0.3543
1,form_strength_model,0.010,None,0.9672,0.5416,0.3530
2,form_strength_model,0.030,None,0.9680,0.5418,0.3520
3,form_strength,0.003,None,0.9683,0.5403,0.3463
4,form_strength_model,0.100,None,0.9685,0.5411,0.3515
5,form_strength_model,0.300,None,0.9686,0.5400,0.3513
6,form_strength_model,1.000,None,0.9687,0.5400,0.3512
7,form_strength,0.010,None,0.9701,0.5389,0.3445
8,form_strength,0.030,None,0.9714,0.5389,0.3435
9,form_strength,0.100,None,0.9720,0.5389,0.3432


## 5. Lätt boostingmodell

Sökningen är avsiktligt liten och regulariserad. Den använder grunda träd, låga learning rates och stark blad-/L2-regularisering. På system utan fungerande XGBoost-runtime används `HistGradientBoostingClassifier`, som är snabb på den längre historiken.

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
BOOSTING_CANDIDATES = 8
boosting_features = FEATURE_SETS["form_strength_model"]
boosting_columns = CATEGORICAL_COLUMNS + boosting_features


def log_uniform(low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))


boosting_configurations = []
for _ in range(BOOSTING_CANDIDATES):
    if NATIVE_XGBOOST:
        boosting_configurations.append({
            "n_estimators": int(rng.integers(120, 321)),
            "max_depth": int(rng.integers(1, 4)),
            "learning_rate": log_uniform(0.01, 0.07),
            "min_child_weight": int(rng.integers(15, 51)),
            "subsample": float(rng.uniform(0.65, 0.90)),
            "colsample_bytree": float(rng.uniform(0.55, 0.85)),
            "reg_alpha": log_uniform(0.1, 5.0),
            "reg_lambda": log_uniform(5.0, 40.0),
            "gamma": float(rng.uniform(0.0, 0.5)),
        })
    else:
        boosting_configurations.append({
            "max_iter": int(rng.integers(80, 181)),
            "max_leaf_nodes": int(rng.choice([7, 11, 15])),
            "max_depth": int(rng.integers(2, 5)),
            "learning_rate": log_uniform(0.01, 0.07),
            "min_samples_leaf": int(rng.integers(50, 151)),
            "l2_regularization": log_uniform(2.0, 25.0),
        })


def boosting_pipeline(configuration):
    if NATIVE_XGBOOST:
        model = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", tree_method="hist",
            random_state=RANDOM_SEED, n_jobs=2, verbosity=0, **configuration,
        )
    else:
        model = HistGradientBoostingClassifier(
            random_state=RANDOM_SEED, **configuration,
        )
    return Pipeline([
        ("preprocessing", preprocessing_pipeline(boosting_features)),
        ("model", model),
    ])


boosting_rows = []
boosting_oof = {}
for candidate, configuration in enumerate(boosting_configurations):
    fold_predictions = []
    for validation_season, train_mask, validation_mask in folds:
        model = boosting_pipeline(configuration)
        model.fit(
            train_data.loc[train_mask, boosting_columns],
            train_data.loc[train_mask, "Target"].map(TARGET_TO_CODE),
        )
        probabilities = probabilities_in_order(
            model, train_data.loc[validation_mask], boosting_columns, encoded_target=True
        )
        fold_predictions.append(pd.DataFrame({
            "Season": validation_season,
            "Target": train_data.loc[validation_mask, "Target"].to_numpy(),
            "ProbabilityLoss": probabilities[:, 0],
            "ProbabilityDraw": probabilities[:, 1],
            "ProbabilityWin": probabilities[:, 2],
        }))
    oof = pd.concat(fold_predictions, ignore_index=True)
    probabilities = oof[["ProbabilityLoss", "ProbabilityDraw", "ProbabilityWin"]].to_numpy()
    predicted = np.asarray(CLASS_ORDER)[probabilities.argmax(axis=1)]
    _, _, gini, _ = macro_roc(oof["Target"], probabilities)
    boosting_oof[candidate] = oof
    boosting_rows.append({
        "Candidate": candidate,
        "CV_LogLoss": log_loss(oof["Target"], probabilities, labels=CLASS_ORDER),
        "CV_Accuracy": accuracy_score(oof["Target"], predicted),
        "CV_MacroGini": gini,
        **configuration,
    })

boosting_tuning = pd.DataFrame(boosting_rows).sort_values(
    ["CV_LogLoss", "CV_MacroGini"], ascending=[True, False]
).reset_index(drop=True)
boosting_best_candidate = int(boosting_tuning.iloc[0]["Candidate"])
boosting_best_configuration = boosting_configurations[boosting_best_candidate]
boosting_best_oof = boosting_oof[boosting_best_candidate]
boosting_oof_probabilities = boosting_best_oof[[
    "ProbabilityLoss", "ProbabilityDraw", "ProbabilityWin",
]].to_numpy()

boosting_model = boosting_pipeline(boosting_best_configuration)
boosting_model.fit(train_data[boosting_columns], train_data["Target"].map(TARGET_TO_CODE))
boosting_train_probabilities = probabilities_in_order(
    boosting_model, train_data, boosting_columns, encoded_target=True
)
boosting_test_probabilities = probabilities_in_order(
    boosting_model, test_data, boosting_columns, encoded_target=True
)
boosting_confidence_threshold = float(np.median(boosting_oof_probabilities.max(axis=1)))

print(f"Boostingmodell: {BOOSTING_MODEL_NAME}")
if not NATIVE_XGBOOST:
    print("Native XGBoost kunde inte laddas (libomp saknas); sklearn-fallback används.")
print("Vald konfiguration:", boosting_best_configuration)
print(f"Säkerhetsgräns: {boosting_confidence_threshold:.3f}")
display(boosting_tuning.head(8).round(4))

Boostingmodell: Sklearn HistGradientBoosting (XGBoost fallback)
Native XGBoost kunde inte laddas (libomp saknas); sklearn-fallback används.
Vald konfiguration: {'max_iter': 127, 'max_leaf_nodes': 11, 'max_depth': 4, 'learning_rate': 0.014626153556787166, 'min_samples_leaf': 86, 'l2_regularization': 8.985542982164532}
Säkerhetsgräns: 0.506


,Candidate,CV_LogLoss,CV_Accuracy,CV_MacroGini,max_iter,max_leaf_nodes,max_depth,learning_rate,min_samples_leaf,l2_regularization
0,6,0.9691,0.5421,0.3499,127,11,4,0.0146,86,8.9855
1,4,0.9692,0.5389,0.3520,146,11,3,0.0172,84,3.5424
2,3,0.9696,0.5403,0.3503,108,11,3,0.0272,126,16.1051
3,1,0.9697,0.5408,0.3499,144,11,4,0.0582,129,3.1302
4,7,0.9698,0.5400,0.3516,112,11,3,0.0150,80,18.2144
5,0,0.9699,0.5405,0.3518,166,7,2,0.0248,114,5.0984
6,5,0.9699,0.5405,0.3514,166,11,2,0.0363,93,2.0659
7,2,0.9705,0.5408,0.3514,166,11,2,0.0656,80,20.4185


## 6. Slutlig utvärdering

`Train OOF` är prognoser från de fem historiska valideringssäsongerna. `Train in-sample` visas bara som kapacitetsdiagnostik. Modelljämförelsen ska baseras på OOF och framför allt den orörda testsäsongen.

Säkerhetsgränsen är respektive modells median-confidence i train-OOF. Samma fasta gräns appliceras på test. Klassvis accuracy redovisas som recall per verkligt utfall.

In [7]:
def predicted_labels(probabilities):
    return np.asarray(CLASS_ORDER)[probabilities.argmax(axis=1)]


logistic_oof_predicted = predicted_labels(logistic_oof_probabilities)
logistic_train_predicted = predicted_labels(logistic_train_probabilities)
logistic_test_predicted = predicted_labels(logistic_test_probabilities)
boosting_oof_predicted = predicted_labels(boosting_oof_probabilities)
boosting_train_predicted = predicted_labels(boosting_train_probabilities)
boosting_test_predicted = predicted_labels(boosting_test_probabilities)

summary_metrics = pd.DataFrame([
    metric_row("Logistic", "Train OOF", logistic_best_oof["Target"], logistic_oof_probabilities),
    metric_row(BOOSTING_MODEL_NAME, "Train OOF", boosting_best_oof["Target"], boosting_oof_probabilities),
    metric_row("Logistic", "Train in-sample", train_data["Target"], logistic_train_probabilities),
    metric_row(BOOSTING_MODEL_NAME, "Train in-sample", train_data["Target"], boosting_train_probabilities),
    metric_row("Logistic", f"Test {TEST_SEASON}", test_data["Target"], logistic_test_probabilities),
    metric_row(BOOSTING_MODEL_NAME, f"Test {TEST_SEASON}", test_data["Target"], boosting_test_probabilities),
])


def confidence_metrics(model_name, split_name, target, probabilities, threshold):
    confidence = probabilities.max(axis=1)
    mask = confidence >= threshold
    row = metric_row(model_name, split_name, np.asarray(target)[mask], probabilities[mask])
    row["Coverage"] = float(mask.mean())
    row["Threshold"] = threshold
    return row, mask


confidence_rows = []
for model_name, oof, oof_probabilities, test_probabilities, threshold in (
    ("Logistic", logistic_best_oof, logistic_oof_probabilities, logistic_test_probabilities, logistic_confidence_threshold),
    (BOOSTING_MODEL_NAME, boosting_best_oof, boosting_oof_probabilities, boosting_test_probabilities, boosting_confidence_threshold),
):
    row, _ = confidence_metrics(model_name, "Train OOF – säkra", oof["Target"], oof_probabilities, threshold)
    confidence_rows.append(row)
    row, _ = confidence_metrics(model_name, f"Test {TEST_SEASON} – säkra", test_data["Target"], test_probabilities, threshold)
    confidence_rows.append(row)
confidence_summary = pd.DataFrame(confidence_rows)


def class_recall_rows(model_name, split_name, target, predicted):
    target = np.asarray(target)
    predicted = np.asarray(predicted)
    rows = []
    for outcome in ("Vinst", "Oavgjort", "Förlust"):
        mask = target == outcome
        rows.append({
            "Model": model_name,
            "Split": split_name,
            "Outcome": outcome,
            "Rows": int(mask.sum()),
            "Recall": float((predicted[mask] == outcome).mean()),
        })
    return rows


class_recall = pd.DataFrame(
    class_recall_rows("Logistic", "Train OOF", logistic_best_oof["Target"], logistic_oof_predicted)
    + class_recall_rows(BOOSTING_MODEL_NAME, "Train OOF", boosting_best_oof["Target"], boosting_oof_predicted)
    + class_recall_rows("Logistic", "Train in-sample", train_data["Target"], logistic_train_predicted)
    + class_recall_rows(BOOSTING_MODEL_NAME, "Train in-sample", train_data["Target"], boosting_train_predicted)
    + class_recall_rows("Logistic", f"Test {TEST_SEASON}", test_data["Target"], logistic_test_predicted)
    + class_recall_rows(BOOSTING_MODEL_NAME, f"Test {TEST_SEASON}", test_data["Target"], boosting_test_predicted)
)

# Negativ kontroll: tre etikettpermutationer med den valda logistiska modellen.
permutation_rows = []
permutation_rng = np.random.default_rng(RANDOM_SEED + 1)
for repeat in range(1, 4):
    shuffled = pd.Series(
        permutation_rng.permutation(train_data["Target"].to_numpy()), index=train_data.index
    )
    fold_frames = []
    for validation_season, train_mask, validation_mask in folds:
        model = Pipeline([
            ("preprocessing", preprocessing_pipeline(logistic_features)),
            ("model", LogisticRegression(
                C=logistic_key[1], class_weight=logistic_key[2], max_iter=5_000,
            )),
        ])
        model.fit(train_data.loc[train_mask, logistic_columns], shuffled.loc[train_mask])
        probabilities = probabilities_in_order(model, train_data.loc[validation_mask], logistic_columns)
        fold_frames.append(pd.DataFrame({
            "Target": train_data.loc[validation_mask, "Target"].to_numpy(),
            "ProbabilityLoss": probabilities[:, 0],
            "ProbabilityDraw": probabilities[:, 1],
            "ProbabilityWin": probabilities[:, 2],
        }))
    permutation_oof = pd.concat(fold_frames, ignore_index=True)
    probabilities = permutation_oof[["ProbabilityLoss", "ProbabilityDraw", "ProbabilityWin"]].to_numpy()
    _, _, gini, _ = macro_roc(permutation_oof["Target"], probabilities)
    permutation_rows.append({"Repeat": repeat, "MacroGini": gini})
permutation_results = pd.DataFrame(permutation_rows)

# Exportera båda modellernas testprediktioner.
test_predictions = test_data[["MatchId", "Season", "MatchDate", "Team", "Opponent", "Venue", "Target"]].copy()
for prefix, probabilities, predicted, threshold in (
    ("Logistic", logistic_test_probabilities, logistic_test_predicted, logistic_confidence_threshold),
    ("Boosting", boosting_test_probabilities, boosting_test_predicted, boosting_confidence_threshold),
):
    test_predictions[f"{prefix}_Predicted"] = predicted
    test_predictions[f"{prefix}_ProbabilityLoss"] = probabilities[:, 0]
    test_predictions[f"{prefix}_ProbabilityDraw"] = probabilities[:, 1]
    test_predictions[f"{prefix}_ProbabilityWin"] = probabilities[:, 2]
    test_predictions[f"{prefix}_Confidence"] = probabilities.max(axis=1)
    test_predictions[f"{prefix}_HighConfidence"] = probabilities.max(axis=1) >= threshold
OUTPUT_PATH = DATA_DIR / "premier_league_all_history_to_2023_24_predictions.csv"
test_predictions.to_csv(OUTPUT_PATH, index=False)


def svg_points(fpr, tpr, left=70, top=55, plot_width=700, plot_height=450):
    return " ".join(
        f"{left + float(x) * plot_width:.1f},{top + (1 - float(y)) * plot_height:.1f}"
        for x, y in zip(fpr, tpr)
    )


def model_curves(oof_target, oof_probabilities, test_probabilities, threshold):
    oof_confidence = oof_probabilities.max(axis=1)
    test_confidence = test_probabilities.max(axis=1)
    oof_mask = oof_confidence >= threshold
    test_mask = test_confidence >= threshold
    all_oof = macro_roc(oof_target, oof_probabilities)
    all_test = macro_roc(test_data["Target"], test_probabilities)
    safe_oof = macro_roc(np.asarray(oof_target)[oof_mask], oof_probabilities[oof_mask])
    safe_test = macro_roc(np.asarray(test_data["Target"])[test_mask], test_probabilities[test_mask])
    return all_oof, all_test, safe_oof, safe_test


logistic_curves = model_curves(
    logistic_best_oof["Target"], logistic_oof_probabilities,
    logistic_test_probabilities, logistic_confidence_threshold,
)
boosting_curves = model_curves(
    boosting_best_oof["Target"], boosting_oof_probabilities,
    boosting_test_probabilities, boosting_confidence_threshold,
)


def roc_svg(title, curves):
    all_oof, all_test, safe_oof, safe_test = curves
    paths = [svg_points(*curve[:2]) for curve in curves]
    ginis = [curve[2] for curve in curves]
    return f"""
    <svg xmlns="http://www.w3.org/2000/svg" width="760" height="540" viewBox="0 0 850 590">
      <rect width="100%" height="100%" fill="white"/>
      <text x="425" y="28" text-anchor="middle" font-family="sans-serif" font-size="19" font-weight="bold">{title}</text>
      <line x1="70" y1="505" x2="770" y2="505" stroke="#333"/>
      <line x1="70" y1="55" x2="70" y2="505" stroke="#333"/>
      <line x1="70" y1="505" x2="770" y2="55" stroke="#888" stroke-width="2" stroke-dasharray="7,7"/>
      <polyline points="{paths[0]}" fill="none" stroke="#2563eb" stroke-width="3"/>
      <polyline points="{paths[1]}" fill="none" stroke="#dc2626" stroke-width="3"/>
      <polyline points="{paths[2]}" fill="none" stroke="#2563eb" stroke-width="3" stroke-dasharray="8,5"/>
      <polyline points="{paths[3]}" fill="none" stroke="#dc2626" stroke-width="3" stroke-dasharray="8,5"/>
      <text x="420" y="550" text-anchor="middle" font-family="sans-serif" font-size="14">False positive rate</text>
      <text x="20" y="280" text-anchor="middle" font-family="sans-serif" font-size="14" transform="rotate(-90 20 280)">True positive rate</text>
      <rect x="400" y="370" width="355" height="112" fill="white" stroke="#ccc"/>
      <text x="415" y="395" font-family="sans-serif" font-size="13" fill="#2563eb">Train OOF – alla: {ginis[0]:.3f}</text>
      <text x="415" y="420" font-family="sans-serif" font-size="13" fill="#2563eb">Train OOF – säkra: {ginis[2]:.3f}</text>
      <text x="415" y="445" font-family="sans-serif" font-size="13" fill="#dc2626">Test – alla: {ginis[1]:.3f}</text>
      <text x="415" y="470" font-family="sans-serif" font-size="13" fill="#dc2626">Test – säkra: {ginis[3]:.3f}</text>
    </svg>
    """


def recall_svg(data):
    colors = {"Vinst": "#16a34a", "Oavgjort": "#d97706", "Förlust": "#dc2626"}
    models = [("Logistic", 70), (BOOSTING_MODEL_NAME, 790)]
    splits = ["Train OOF", "Train in-sample", f"Test {TEST_SEASON}"]
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="1500" height="520" viewBox="0 0 1500 520">', '<rect width="100%" height="100%" fill="white"/>', '<text x="750" y="28" text-anchor="middle" font-family="sans-serif" font-size="21" font-weight="bold">Klassvis recall</text>']
    for index, outcome in enumerate(("Vinst", "Oavgjort", "Förlust")):
        x = 570 + index * 145
        parts += [f'<rect x="{x}" y="43" width="16" height="16" fill="{colors[outcome]}"/>', f'<text x="{x + 22}" y="56" font-family="sans-serif" font-size="13">{outcome}</text>']
    for model_name, left in models:
        right, top, bottom = left + 630, 90, 410
        parts += [f'<text x="{(left + right) / 2:.0f}" y="82" text-anchor="middle" font-family="sans-serif" font-size="16" font-weight="bold">{model_name}</text>', f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#333"/>', f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/>']
        for tick in (0, 0.25, 0.5, 0.75, 1):
            y = bottom - tick * (bottom - top)
            parts += [f'<line x1="{left}" y1="{y}" x2="{right}" y2="{y}" stroke="#e5e7eb"/>', f'<text x="{left - 8}" y="{y + 4}" text-anchor="end" font-family="sans-serif" font-size="11">{tick:.2f}</text>']
        for split_index, split in enumerate(splits):
            center = left + 115 + split_index * 205
            parts.append(f'<text x="{center}" y="435" text-anchor="middle" font-family="sans-serif" font-size="12">{split}</text>')
            for outcome_index, outcome in enumerate(("Vinst", "Oavgjort", "Förlust")):
                value = float(data.loc[(data.Model == model_name) & (data.Split == split) & (data.Outcome == outcome), "Recall"].iloc[0])
                x = center - 48 + outcome_index * 38
                bar_height = value * (bottom - top)
                y = bottom - bar_height
                parts += [f'<rect x="{x}" y="{y:.1f}" width="30" height="{bar_height:.1f}" fill="{colors[outcome]}"/>', f'<text x="{x + 15}" y="{max(top + 10, y - 5):.1f}" text-anchor="middle" font-family="sans-serif" font-size="10">{value:.2f}</text>']
    parts.append('</svg>')
    return "".join(parts)


logistic_svg = roc_svg("Logistisk regression – all historik", logistic_curves)
boosting_svg = roc_svg(f"{BOOSTING_MODEL_NAME} – all historik", boosting_curves)
class_svg = recall_svg(class_recall)
plots_html = f"""
<div style="display:flex; gap:12px; overflow-x:auto">
  <div style="min-width:700px; flex:1">{logistic_svg}</div>
  <div style="min-width:700px; flex:1">{boosting_svg}</div>
</div>
<div style="margin-top:18px; overflow-x:auto">{class_svg}</div>
"""

print(f"Trainperiod: {train_data.Season.min()}–{train_data.Season.max()} ({len(train_data):,} lagrader)")
print(f"Testperiod: {TEST_SEASON} ({len(test_data):,} lagrader)")
print(f"Sparad: {OUTPUT_PATH}")
print("\nSamtliga mått:")
display(summary_metrics.round(3))
print("\nSäkra prognoser:")
display(confidence_summary.round(3))
print("\nKlassvis recall:")
display(class_recall.round(3))
print("\nNegativ kontroll – permuterade train-labels:")
display(permutation_results.round(3))
display({"text/html": plots_html}, raw=True)

Trainperiod: 2000/01–2022/23 (17,480 lagrader)
Testperiod: 2023/24 (760 lagrader)
Sparad: /Users/andreasnilsson/Desktop/Repos/betting-optimizer/data/premier_league_all_history_to_2023_24_predictions.csv

Samtliga mått:


,Model,Split,Rows,Accuracy,LogLoss,MacroGini,Gini_Förlust,Gini_Oavgjort,Gini_Vinst
0,Logistic,Train OOF,3800,0.544,0.966,0.354,0.459,0.142,0.461
1,Sklearn HistGradientBoosting (XGBoost fallback),Train OOF,3800,0.542,0.969,0.350,0.455,0.140,0.454
2,Logistic,Train in-sample,17480,0.534,0.974,0.361,0.466,0.148,0.467
3,Sklearn HistGradientBoosting (XGBoost fallback),Train in-sample,17480,0.536,0.966,0.392,0.478,0.216,0.480
4,Logistic,Test 2023/24,760,0.580,0.927,0.391,0.526,0.108,0.538
5,Sklearn HistGradientBoosting (XGBoost fallback),Test 2023/24,760,0.567,0.938,0.385,0.515,0.123,0.517



Säkra prognoser:


,Model,Split,Rows,Accuracy,LogLoss,MacroGini,Gini_Förlust,Gini_Oavgjort,Gini_Vinst,Coverage,Threshold
0,Logistic,Train OOF – säkra,1900,0.663,0.856,0.463,0.602,0.177,0.611,0.500,0.518
1,Logistic,Test 2023/24 – säkra,410,0.680,0.813,0.515,0.663,0.196,0.687,0.539,0.518
2,Sklearn HistGradientBoosting (XGBoost fallback),Train OOF – säkra,1900,0.658,0.861,0.457,0.600,0.171,0.601,0.500,0.506
3,Sklearn HistGradientBoosting (XGBoost fallback),Test 2023/24 – säkra,398,0.668,0.830,0.507,0.671,0.186,0.664,0.524,0.506



Klassvis recall:


,Model,Split,Outcome,Rows,Recall
0,Logistic,Train OOF,Vinst,1479,0.686
1,Logistic,Train OOF,Oavgjort,842,0.000
2,Logistic,Train OOF,Förlust,1479,0.711
3,Sklearn HistGradientBoosting (XGBoost fallback),Train OOF,Vinst,1479,0.697
4,Sklearn HistGradientBoosting (XGBoost fallback),Train OOF,Oavgjort,842,0.000
5,Sklearn HistGradientBoosting (XGBoost fallback),Train OOF,Förlust,1479,0.696
6,Logistic,Train in-sample,Vinst,6568,0.705
7,Logistic,Train in-sample,Oavgjort,4344,0.000
8,Logistic,Train in-sample,Förlust,6568,0.717
9,Sklearn HistGradientBoosting (XGBoost fallback),Train in-sample,Vinst,6568,0.710



Negativ kontroll – permuterade train-labels:


,Repeat,MacroGini
0,1,0.139
1,2,0.007
2,3,-0.155


Logistisk regression – all historik 
 
 
 
 <polyline points="70.0,505.0 71.4,495.7 72.8,492.7 74.2,490.2 75.6,482.8 77.0,479.0 78.4,476.6 79.8,472.3 81.2,469.1 82.6,467.1 84.0,464.6 85.4,462.2 86.8,459.4 88.2,456.1 89.6,454.0 91.0,450.3 92.4,449.5 93.8,446.7 95.2,444.3 96.6,442.6 98.0,440.9 99.4,438.6 100.8,437.1 102.2,433.0 103.6,430.7 105.0,428.1 106.4,426.4 107.8,423.2 109.2,421.0 110.6,419.2 112.0,417.0 113.4,415.0 114.8,413.2 116.2,411.0 117.6,409.5 119.0,407.8 120.4,406.6 121.8,405.0 123.2,403.1 124.6,402.0 126.0,400.8 127.4,397.9 128.8,396.7 130.2,394.5 131.6,392.5 133.0,391.9 134.4,390.6 135.8,389.6 137.2,389.0 138.6,386.8 140.0,383.9 141.4,381.9 142.8,380.3 144.2,378.0 145.6,377.0 147.0,375.5 148.4,374.8 149.8,373.4 151.2,372.4 152.6,371.1 154.0,369.7 155.4,366.3 156.8,364.4 158.2,363.5 159.6,362.2 161.0,360.7 162.4,358.8 163.8,357.6 165.2,356.8 166.6,355.4 168.0,353.9 169.4,351.6 170.8,350.0 172.2,349.0 173.6,348.3 175.0,347.1 176.4,346.4 177.8,345.3 179.2,343.7 180.6,343.0 182.0,342.2 183.4,340.4 184.8,338.0 186.2,336.1 187.6,334.9 189.0,333.2 190.4,330.6 191.8,329.4 193.2,328.8 194.6,327.9 196.0,327.2 197.4,325.9 198.8,325.4 200.2,324.7 201.6,323.9 203.0,322.3 204.4,321.3 205.8,319.8 207.2,318.6 208.6,318.3 210.0,317.0 211.4,315.7 212.8,314.3 214.2,312.6 215.6,311.8 217.0,310.5 218.4,308.3 219.8,307.3 221.2,305.8 222.6,305.5 224.0,304.6 225.4,303.7 226.8,303.3 228.2,302.2 229.6,301.4 231.0,300.1 232.4,299.5 233.8,298.2 235.2,297.1 236.6,295.8 238.0,294.7 239.4,294.1 240.8,292.8 242.2,292.7 243.6,291.5 245.0,290.0 246.4,288.8 247.8,287.9 249.2,287.3 250.6,285.8 252.0,284.5 253.4,283.8 254.8,283.5 256.2,281.2 257.6,280.7 259.0,279.6 260.4,279.2 261.8,278.4 263.2,277.7 264.6,276.2 266.0,274.8 267.4,274.1 268.8,272.9 270.2,272.0 271.6,270.9 273.0,270.4 274.4,269.2 275.8,268.2 277.2,267.2 278.6,265.7 280.0,264.2 281.4,263.0 282.8,261.9 284.2,260.2 285.6,259.6 287.0,258.0 288.4,256.6 289.8,255.4 291.2,254.4 292.6,253.5 294.0,252.7 295.4,252.1 296.8,251.1 298.2,250.5 299.6,248.1 301.0,247.2 302.4,246.1 303.8,245.6 305.2,245.0 306.6,244.5 308.0,242.9 309.4,242.0 310.8,241.1 312.2,240.0 313.6,239.1 315.0,238.3 316.4,237.3 317.8,236.8 319.2,236.1 320.6,235.6 322.0,234.2 323.4,233.7 324.8,232.9 326.2,232.3 327.6,231.6 329.0,229.9 330.4,229.1 331.8,228.5 333.2,226.8 334.6,225.5 336.0,224.9 337.4,224.2 338.8,223.1 340.2,222.6 341.6,222.1 343.0,220.7 344.4,219.7 345.8,219.1 347.2,217.2 348.6,216.0 350.0,215.2 351.4,214.3 352.8,213.1 354.2,212.5 355.6,211.8 357.0,210.9 358.4,209.3 359.8,207.4 361.2,207.1 362.6,206.2 364.0,205.6 365.4,204.7 366.8,203.9 368.2,202.9 369.6,202.1 371.0,201.0 372.4,199.7 373.8,198.9 375.2,197.8 376.6,196.5 378.0,195.8 379.4,195.4 380.8,194.1 382.2,192.8 383.6,191.9 385.0,191.6 386.4,190.5 387.8,190.1 389.2,188.8 390.6,188.2 392.0,187.9 393.4,187.2 394.8,186.1 396.2,184.3 397.6,183.4 399.0,182.8 400.4,181.7 401.8,180.6 403.2,179.7 404.6,178.9 406.0,177.8 407.4,176.7 408.8,176.2 410.2,175.6 411.6,174.7 413.0,174.1 414.4,173.3 415.8,172.1 417.2,171.2 418.6,170.5 420.0,169.9 421.4,168.9 422.8,168.1 424.2,167.0 425.6,166.2 427.0,165.6 428.4,165.1 429.8,163.5 431.2,163.0 432.6,161.9 434.0,161.6 435.4,161.3 436.8,159.9 438.2,159.2 439.6,158.7 441.0,158.1 442.4,157.4 443.8,157.2 445.2,156.3 446.6,155.9 448.0,155.4 449.4,155.2 450.8,154.3 452.2,153.6 453.6,152.9 455.0,152.8 456.4,152.1 457.8,151.8 459.2,151.2 460.6,150.7 462.0,150.5 463.4,149.9 464.8,149.6 466.2,149.2 467.6,148.4 469.0,148.0 470.4,147.2 471.8,145.9 473.2,145.8 474.6,145.1 476.0,143.5 477.4,142.8 478.8,142.1 480.2,140.7 481.6,140.1 483.0,139.6 484.4,137.8 485.8,137.6 487.2,137.2 488.6,136.9 490.0,135.9 491.4,135.7 492.8,135.4 494.2,135.0 495.6,134.8 497.0,134.6 498.4,134.3 499.8,133.0 501.2,132.0 502.6,129.9 504.0,129.4 505.4,128.5 506.8,127.9 508.2,127.4 509.6,126.9 511.0,125.7 512.4,125.1 513.8,124.4 515.2,124.2 516.6,123.5 518.0,122.9 519.4,121.7 520.8,121.6 522.2,121.1 523.6,119.8 525.0,119.2 526.4,118.8 527.8,118.0 529.2,117.8 530.6,116.9 